# Modal remote functions

This notebook stays local in Cursor. Only the decorated function runs remotely on Modal.

Run `uv sync --extra remote` and `uv run modal setup` once before executing it.

In [ ]:
import modal

app = modal.App("ml-lab-notebook")
image = modal.Image.debian_slim(python_version="3.12").pip_install("torch")

In [ ]:
@app.function(image=image, gpu="T4", timeout=300)
def remote_matmul(size: int = 1024) -> dict[str, object]:
    import torch

    left = torch.randn(size, size, device="cuda")
    right = torch.randn(size, size, device="cuda")
    result = left @ right
    return {
        "device": str(result.device),
        "gpu": torch.cuda.get_device_name(0),
        "shape": list(result.shape),
    }

In [ ]:
with modal.enable_output():
    with app.run():
        result = remote_matmul.remote(1024)

print(result)